In [17]:
from pathlib import Path

from scipy.optimize import linear_sum_assignment
import torch

In [ ]:
TORCH_SEED = 42
BATCH_SIZE = 32
NUM_CLASSES = 3  # background + rectangle + circle
MAX_NUM_OBJS = 10
IMAGE_SIZE = 128

BBOX_COST_WEIGHT = 1.0
CLASS_COST_WEIGHT = 2.0

In [19]:
x, target = torch.load("batch.pt", weights_only=False)
x.shape, target["boxes"].shape, target["labels"].shape, target["object_mask"].shape

(torch.Size([32, 3, 128, 128]),
 torch.Size([32, 3, 4]),
 torch.Size([32, 3]),
 torch.Size([32, 3]))

In [20]:
target_boxes = target["boxes"]
target_boxes_normalized = target_boxes / IMAGE_SIZE

target_labels = target["labels"]
object_mask = target["object_mask"]

In [21]:
torch.manual_seed(TORCH_SEED)

pred_locations = torch.rand(BATCH_SIZE, MAX_NUM_OBJS, 4)

pred_class_scores = torch.randn(BATCH_SIZE, MAX_NUM_OBJS, NUM_CLASSES)
pred_class_probs = torch.softmax(pred_class_scores, dim=-1)

pred_locations.shape, pred_class_probs.shape

(torch.Size([32, 10, 4]), torch.Size([32, 10, 3]))

# bbox cost

In [22]:
pred_locations.shape, pred_locations[0]

(torch.Size([32, 10, 4]),
 tensor([[0.8823, 0.9150, 0.3829, 0.9593],
         [0.3904, 0.6009, 0.2566, 0.7936],
         [0.9408, 0.1332, 0.9346, 0.5936],
         [0.8694, 0.5677, 0.7411, 0.4294],
         [0.8854, 0.5739, 0.2666, 0.6274],
         [0.2696, 0.4414, 0.2969, 0.8317],
         [0.1053, 0.2695, 0.3588, 0.1994],
         [0.5472, 0.0062, 0.9516, 0.0753],
         [0.8860, 0.5832, 0.3376, 0.8090],
         [0.5779, 0.9040, 0.5547, 0.3423]]))

In [23]:
target_boxes.shape, target_boxes[0]

(torch.Size([32, 3, 4]),
 tensor([[41., 42., 61., 62.],
         [19., 66., 43., 78.],
         [ 0.,  0.,  0.,  0.]]))

In [24]:
# Compute pairwise costs against every padded target slot first.
bbox_cost = torch.cdist(pred_locations, target_boxes_normalized, p=2)
bbox_cost.shape

torch.Size([32, 10, 3])

# class cost

In [25]:
pred_class_probs.shape, pred_class_probs[0]

(torch.Size([32, 10, 3]),
 tensor([[0.1671, 0.1696, 0.6633],
         [0.3678, 0.5164, 0.1159],
         [0.6159, 0.2579, 0.1262],
         [0.2270, 0.4838, 0.2891],
         [0.1415, 0.8241, 0.0344],
         [0.2844, 0.3465, 0.3691],
         [0.3836, 0.5248, 0.0915],
         [0.0589, 0.1124, 0.8287],
         [0.5719, 0.2890, 0.1391],
         [0.6762, 0.1609, 0.1629]]))

In [26]:
target_labels.shape, target_labels[0]

(torch.Size([32, 3]), tensor([2, 1, 0]))

In [27]:
import torch.nn.functional as F

target_labels_one_hot = F.one_hot(target_labels, num_classes=NUM_CLASSES).float()
target_labels_one_hot.shape, target_labels_one_hot[0]

(torch.Size([32, 3, 3]),
 tensor([[0., 0., 1.],
         [0., 1., 0.],
         [1., 0., 0.]]))

In [28]:
class_cost = torch.cdist(
    pred_class_probs,
    target_labels_one_hot,
    p=2,
)

class_cost.shape

torch.Size([32, 10, 3])

In [29]:
bbox_cost.min(), bbox_cost.max(), class_cost.min(), class_cost.max()

(tensor(0.1284), tensor(1.8310), tensor(0.0985), tensor(1.3737))

# agg cost

In [30]:
cost = bbox_cost * BBOX_COST_WEIGHT + class_cost * CLASS_COST_WEIGHT

In [31]:
cost.shape

torch.Size([32, 10, 3])

In [14]:
label_indices = target_labels.unsqueeze(1).expand(-1, pred_class_probs.shape[1], -1)
class_cost = -pred_class_probs.gather(dim=2, index=label_indices)
class_cost.shape

torch.Size([32, 10, 3])

In [15]:
pairwise_cost = bbox_cost_weight * bbox_cost + class_cost_weight * class_cost

NameError: name 'bbox_cost_weight' is not defined

In [ ]:


    # Only after the pairwise distances/costs exist, mask padded target slots.
    masked_pairwise_cost = pairwise_cost.masked_fill(
        ~object_mask.unsqueeze(1), torch.inf
    )

    matches = []
    for batch_idx in range(masked_pairwise_cost.shape[0]):
        valid_target_indices = object_mask[batch_idx].nonzero(as_tuple=True)[0]

        if valid_target_indices.numel() == 0:
            matches.append(
                (
                    torch.empty(0, dtype=torch.long),
                    torch.empty(0, dtype=torch.long),
                )
            )
            continue

        sample_cost = masked_pairwise_cost[batch_idx, :, valid_target_indices]
        pred_indices, valid_target_positions = linear_sum_assignment(
            sample_cost.cpu().numpy()
        )

        matches.append(
            (
                torch.as_tensor(pred_indices, dtype=torch.long),
                valid_target_indices[
                    torch.as_tensor(valid_target_positions, dtype=torch.long)
                ].cpu(),
            )
        )

    return matches, pairwise_cost, masked_pairwise_cost


matches, pairwise_cost, masked_pairwise_cost = hungarian_match(
    pred_locations=pred_locations,
    pred_class_probs=pred_class_probs,
    target=target,
)

matches[0], pairwise_cost.shape, masked_pairwise_cost.shape